# Spark et Databricks (Partie 4) : agrégations, jointures et fenêtres

📖 **Le mini-cours complet** : [Spark et Databricks : agrégations, jointures et fonctions de fenêtre en PySpark (Partie 4)](https://www.monshotdata.com/p/spark-databricks-agregations-jointures-fenetres-partie-4)

🧭 **Le cours** : [Bonnes pratiques d'ingénierie › 1.4 Passage à l'échelle et calcul distribué](https://www.monshotdata.com/mini-cours/bonnes-pratiques-ingenierie#h2sc14)

---

Résumer des données par groupe, croiser deux tables, calculer une moyenne mobile ou un classement : ce sont les opérations qui transforment des données brutes en analyse. Ici tu les réalises toutes en PySpark, puis tu traites les valeurs manquantes.

**À la fin de ce notebook, tu sauras :**

- agréger avec `groupBy()` et `agg()`, et construire un tableau croisé avec `pivot()` ;
- utiliser les 7 types de jointures, et concaténer des DataFrames avec `unionByName()` ;
- écrire des fonctions de fenêtre : moyenne mobile, classements, `lag()`, `lead()` ;
- compter, supprimer et remplir les valeurs manquantes avec `na`.

### Avant de lancer

Ce notebook tourne sur **Databricks**, pas sur Colab. Un compte **Databricks Free Edition** (gratuit) suffit.

- **Ouvrir le notebook** : dans Databricks, Workspace → Import, puis dépose ce fichier `.ipynb` (ou colle l'URL du fichier sur GitHub).
- **Le lancer** : attache-le au calcul **serverless**, puis exécute les cellules de haut en bas (ou « Run all »).
- **Les données** : elles sont toutes créées dans le notebook : rien à préparer, et aucun fichier n'est écrit.

🧪 *Testé le 21 novembre 2026 : exécution complète de haut en bas sur Databricks Free Edition (calcul serverless, Spark 4.2, Python 3.12). Les sorties affichées viennent de cette exécution ; les identifiants techniques (adresse de la session, noms de fichiers `part-…`, `uid` des modèles) changent à chaque lancement. Deux cellules de fenêtre sans `partitionBy()` affichent un avertissement `UserWarning` de Spark : il est normal, et il illustre justement le propos de la cellule.*

## 1. Agréger et croiser : groupBy et pivot

Agrégations par groupe, sans groupe, et tableau croisé avec `pivot()`.

In [1]:
# Données de départ
from pyspark.sql import functions as F

donnees = [
    ("Paris", "Manager", 34, 80000, 7.5),
    ("Lyon", "Ingénieur", 28, 70000, 6.0),
    ("Marseille", "Analyste", 25, 60000, 8.2),
    ("Paris", "Ingénieur", 28, 75000, 5.1),
    ("Lyon", "Manager", 41, 90000, 9.4),
    ("Marseille", "Ingénieur", 25, 65000, 4.8),
    ("Paris", "Analyste", 34, 62000, 6.6),
    ("Lyon", "Analyste", 28, 58000, 7.9),
    ("Paris", "Ingénieur", 31, 79000, 6.2),
]
schema = "ville STRING, poste STRING, age INT, salaire INT, credit DOUBLE"
df = spark.createDataFrame(donnees, schema)
df.show()
# +---------+---------+---+-------+------+
# |    ville|    poste|age|salaire|credit|
# +---------+---------+---+-------+------+
# |    Paris|  Manager| 34|  80000|   7.5|
# |     Lyon|Ingénieur| 28|  70000|   6.0|
# |Marseille| Analyste| 25|  60000|   8.2|
# |    Paris|Ingénieur| 28|  75000|   5.1|
# |     Lyon|  Manager| 41|  90000|   9.4|
# |Marseille|Ingénieur| 25|  65000|   4.8|
# |    Paris| Analyste| 34|  62000|   6.6|
# |     Lyon| Analyste| 28|  58000|   7.9|
# |    Paris|Ingénieur| 31|  79000|   6.2|
# +---------+---------+---+-------+------+

+---------+---------+---+-------+------+
|    ville|    poste|age|salaire|credit|
+---------+---------+---+-------+------+
|    Paris|  Manager| 34|  80000|   7.5|
|     Lyon|Ingénieur| 28|  70000|   6.0|
|Marseille| Analyste| 25|  60000|   8.2|
|    Paris|Ingénieur| 28|  75000|   5.1|
|     Lyon|  Manager| 41|  90000|   9.4|
|Marseille|Ingénieur| 25|  65000|   4.8|
|    Paris| Analyste| 34|  62000|   6.6|
|     Lyon| Analyste| 28|  58000|   7.9|
|    Paris|Ingénieur| 31|  79000|   6.2|
+---------+---------+---+-------+------+



In [2]:
# Une seule agrégation par groupe
groupes = df.groupBy("ville")
print(type(groupes))
# <class 'pyspark.sql.connect.group.GroupedData'>

groupes.agg(F.avg("salaire")).orderBy("ville").show()
# +---------+-----------------+
# |    ville|     avg(salaire)|
# +---------+-----------------+
# |     Lyon|72666.66666666667|
# |Marseille|          62500.0|
# |    Paris|          74000.0|
# +---------+-----------------+

# Fréquence de chaque ville (l'équivalent de value_counts() en Pandas)
df.groupBy("ville").count().orderBy("ville").show()
# +---------+-----+
# |    ville|count|
# +---------+-----+
# |     Lyon|    3|
# |Marseille|    2|
# |    Paris|    4|
# +---------+-----+

<class 'pyspark.sql.connect.group.GroupedData'>
+---------+-----------------+
|    ville|     avg(salaire)|
+---------+-----------------+
|     Lyon|72666.66666666667|
|Marseille|          62500.0|
|    Paris|          74000.0|
+---------+-----------------+

+---------+-----+
|    ville|count|
+---------+-----+
|     Lyon|    3|
|Marseille|    2|
|    Paris|    4|
+---------+-----+



In [3]:
# Plusieurs agrégations sur le même groupe
(df.groupBy("ville")
   .agg(F.avg("salaire").alias("salaire_moyen"),
        F.max("credit").alias("credit_max"))
   .orderBy("ville")
   .show())
# +---------+-----------------+----------+
# |    ville|    salaire_moyen|credit_max|
# +---------+-----------------+----------+
# |     Lyon|72666.66666666667|       9.4|
# |Marseille|          62500.0|       8.2|
# |    Paris|          74000.0|       7.5|
# +---------+-----------------+----------+

# Grouper sur plusieurs colonnes
(df.groupBy("ville", "age")
   .agg(F.count("*").alias("effectif"))
   .orderBy("ville", "age")
   .show())
# +---------+---+--------+
# |    ville|age|effectif|
# +---------+---+--------+
# |     Lyon| 28|       2|
# |     Lyon| 41|       1|
# |Marseille| 25|       2|
# |    Paris| 28|       1|
# |    Paris| 31|       1|
# |    Paris| 34|       2|
# +---------+---+--------+

+---------+-----------------+----------+
|    ville|    salaire_moyen|credit_max|
+---------+-----------------+----------+
|     Lyon|72666.66666666667|       9.4|
|Marseille|          62500.0|       8.2|
|    Paris|          74000.0|       7.5|
+---------+-----------------+----------+

+---------+---+--------+
|    ville|age|effectif|
+---------+---+--------+
|     Lyon| 28|       2|
|     Lyon| 41|       1|
|Marseille| 25|       2|
|    Paris| 28|       1|
|    Paris| 31|       1|
|    Paris| 34|       2|
+---------+---+--------+



In [4]:
# Agréger sans groupBy : tout le DataFrame forme un seul groupe
df.select(F.avg("salaire")).show()
# +------------+
# |avg(salaire)|
# +------------+
# |     71000.0|
# +------------+

df.agg(F.min("salaire").alias("salaire_min"),
       F.max("salaire").alias("salaire_max"),
       F.sum("salaire").alias("masse_salariale")).show()
# +-----------+-----------+---------------+
# |salaire_min|salaire_max|masse_salariale|
# +-----------+-----------+---------------+
# |      58000|      90000|         639000|
# +-----------+-----------+---------------+

+------------+
|avg(salaire)|
+------------+
|     71000.0|
+------------+

+-----------+-----------+---------------+
|salaire_min|salaire_max|masse_salariale|
+-----------+-----------+---------------+
|      58000|      90000|         639000|
+-----------+-----------+---------------+



In [5]:
# Tableau croisé : villes en lignes, postes en colonnes, salaire moyen par case
(df.groupBy("ville")
   .pivot("poste")
   .agg(F.avg("salaire"))
   .orderBy("ville")
   .show())
# +---------+--------+---------+-------+
# |    ville|Analyste|Ingénieur|Manager|
# +---------+--------+---------+-------+
# |     Lyon| 58000.0|  70000.0|90000.0|
# |Marseille| 60000.0|  65000.0|   NULL|
# |    Paris| 62000.0|  77000.0|80000.0|
# +---------+--------+---------+-------+

# Plus rapide : tu fournis les valeurs du pivot, Spark n'a pas à les calculer
(df.groupBy("ville")
   .pivot("poste", ["Analyste", "Ingénieur", "Manager"])
   .agg(F.avg("salaire"))
   .orderBy("ville")
   .show())
# +---------+--------+---------+-------+
# |    ville|Analyste|Ingénieur|Manager|
# +---------+--------+---------+-------+
# |     Lyon| 58000.0|  70000.0|90000.0|
# |Marseille| 60000.0|  65000.0|   NULL|
# |    Paris| 62000.0|  77000.0|80000.0|
# +---------+--------+---------+-------+

+---------+--------+---------+-------+
|    ville|Analyste|Ingénieur|Manager|
+---------+--------+---------+-------+
|     Lyon| 58000.0|  70000.0|90000.0|
|Marseille| 60000.0|  65000.0|   NULL|
|    Paris| 62000.0|  77000.0|80000.0|
+---------+--------+---------+-------+

+---------+--------+---------+-------+
|    ville|Analyste|Ingénieur|Manager|
+---------+--------+---------+-------+
|     Lyon| 58000.0|  70000.0|90000.0|
|Marseille| 60000.0|  65000.0|   NULL|
|    Paris| 62000.0|  77000.0|80000.0|
+---------+--------+---------+-------+



## 2. Joindre des DataFrames

Les deux DataFrames à joindre, puis la jointure interne.

In [6]:
# Deux DataFrames à joindre
donnees_gauche = [
    ("Paris", "Manager", 80000),
    ("Lyon", "Ingénieur", 70000),
    ("Marseille", "Analyste", 60000),
    ("Toulouse", "Manager", 90000),
    ("Marseille", "Ingénieur", 75000),
    ("Paris", "Stagiaire", 32000),
]
df_gauche = spark.createDataFrame(donnees_gauche,
                                  "ville STRING, poste STRING, salaire INT")

donnees_droite = [("Directeur", "N4"), ("Manager", "N3"),
                  ("Analyste", "N2"), ("Ingénieur", "N1")]
df_droite = spark.createDataFrame(donnees_droite, "poste STRING, niveau STRING")

df_gauche.show()
# +---------+---------+-------+
# |    ville|    poste|salaire|
# +---------+---------+-------+
# |    Paris|  Manager|  80000|
# |     Lyon|Ingénieur|  70000|
# |Marseille| Analyste|  60000|
# | Toulouse|  Manager|  90000|
# |Marseille|Ingénieur|  75000|
# |    Paris|Stagiaire|  32000|
# +---------+---------+-------+
df_droite.show()
# +---------+------+
# |    poste|niveau|
# +---------+------+
# |Directeur|    N4|
# |  Manager|    N3|
# | Analyste|    N2|
# |Ingénieur|    N1|
# +---------+------+

+---------+---------+-------+
|    ville|    poste|salaire|
+---------+---------+-------+
|    Paris|  Manager|  80000|
|     Lyon|Ingénieur|  70000|
|Marseille| Analyste|  60000|
| Toulouse|  Manager|  90000|
|Marseille|Ingénieur|  75000|
|    Paris|Stagiaire|  32000|
+---------+---------+-------+

+---------+------+
|    poste|niveau|
+---------+------+
|Directeur|    N4|
|  Manager|    N3|
| Analyste|    N2|
|Ingénieur|    N1|
+---------+------+



In [7]:
# Jointure interne (how="inner" par défaut)
df_gauche.join(df_droite, on="poste").show()
# +---------+---------+-------+------+
# |    poste|    ville|salaire|niveau|
# +---------+---------+-------+------+
# |  Manager|    Paris|  80000|    N3|
# |Ingénieur|     Lyon|  70000|    N1|
# | Analyste|Marseille|  60000|    N2|
# |  Manager| Toulouse|  90000|    N3|
# |Ingénieur|Marseille|  75000|    N1|
# +---------+---------+-------+------+

# La même jointure avec une expression de colonne
df_gauche.join(df_droite, on=df_gauche.poste == df_droite.poste).show()
# +---------+---------+-------+---------+------+
# |    ville|    poste|salaire|    poste|niveau|
# +---------+---------+-------+---------+------+
# |    Paris|  Manager|  80000|  Manager|    N3|
# |     Lyon|Ingénieur|  70000|Ingénieur|    N1|
# |Marseille| Analyste|  60000| Analyste|    N2|
# | Toulouse|  Manager|  90000|  Manager|    N3|
# |Marseille|Ingénieur|  75000|Ingénieur|    N1|
# +---------+---------+-------+---------+------+

+---------+---------+-------+------+
|    poste|    ville|salaire|niveau|
+---------+---------+-------+------+
|  Manager|    Paris|  80000|    N3|
|Ingénieur|     Lyon|  70000|    N1|
| Analyste|Marseille|  60000|    N2|
|  Manager| Toulouse|  90000|    N3|
|Ingénieur|Marseille|  75000|    N1|
+---------+---------+-------+------+

+---------+---------+-------+---------+------+
|    ville|    poste|salaire|    poste|niveau|
+---------+---------+-------+---------+------+
|    Paris|  Manager|  80000|  Manager|    N3|
|     Lyon|Ingénieur|  70000|Ingénieur|    N1|
|Marseille| Analyste|  60000| Analyste|    N2|
| Toulouse|  Manager|  90000|  Manager|    N3|
|Marseille|Ingénieur|  75000|Ingénieur|    N1|
+---------+---------+-------+---------+------+



In [8]:
# La clé ne porte pas le même nom des
# deux côtés : pas besoin de renommer
df_fonctions = df_droite.withColumnRenamed("poste", "fonction")

(df_gauche
   .join(df_fonctions, on=df_gauche.poste == df_fonctions.fonction)
   .drop("fonction")
   .show())
# +---------+---------+-------+------+
# |    ville|    poste|salaire|niveau|
# +---------+---------+-------+------+
# |    Paris|  Manager|  80000|    N3|
# |     Lyon|Ingénieur|  70000|    N1|
# |Marseille| Analyste|  60000|    N2|
# | Toulouse|  Manager|  90000|    N3|
# |Marseille|Ingénieur|  75000|    N1|
# +---------+---------+-------+------+

+---------+---------+-------+------+
|    ville|    poste|salaire|niveau|
+---------+---------+-------+------+
|    Paris|  Manager|  80000|    N3|
|     Lyon|Ingénieur|  70000|    N1|
|Marseille| Analyste|  60000|    N2|
| Toulouse|  Manager|  90000|    N3|
|Marseille|Ingénieur|  75000|    N1|
+---------+---------+-------+------+



## 3. Les autres types de jointures

Jointures gauche, externe complète, semi, anti et croisée.

In [9]:
# Jointure gauche : toutes les lignes
# de gauche, NULL sans correspondance
df_gauche.join(df_droite, on="poste", how="left").show()
# +---------+---------+-------+------+
# |    poste|    ville|salaire|niveau|
# +---------+---------+-------+------+
# |  Manager|    Paris|  80000|    N3|
# |Ingénieur|     Lyon|  70000|    N1|
# | Analyste|Marseille|  60000|    N2|
# |  Manager| Toulouse|  90000|    N3|
# |Ingénieur|Marseille|  75000|    N1|
# |Stagiaire|    Paris|  32000|  NULL|
# +---------+---------+-------+------+

+---------+---------+-------+------+
|    poste|    ville|salaire|niveau|
+---------+---------+-------+------+
|  Manager|    Paris|  80000|    N3|
|Ingénieur|     Lyon|  70000|    N1|
| Analyste|Marseille|  60000|    N2|
|  Manager| Toulouse|  90000|    N3|
|Ingénieur|Marseille|  75000|    N1|
|Stagiaire|    Paris|  32000|  NULL|
+---------+---------+-------+------+



In [10]:
# Jointure externe complète : toutes les lignes des deux côtés
df_gauche.join(df_droite, on="poste", how="full").show()
# +---------+---------+-------+------+
# |    poste|    ville|salaire|niveau|
# +---------+---------+-------+------+
# |  Manager|    Paris|  80000|    N3|
# |Ingénieur|     Lyon|  70000|    N1|
# | Analyste|Marseille|  60000|    N2|
# |  Manager| Toulouse|  90000|    N3|
# |Ingénieur|Marseille|  75000|    N1|
# |Stagiaire|    Paris|  32000|  NULL|
# |Directeur|     NULL|   NULL|    N4|
# +---------+---------+-------+------+

+---------+---------+-------+------+
|    poste|    ville|salaire|niveau|
+---------+---------+-------+------+
|  Manager|    Paris|  80000|    N3|
|Ingénieur|     Lyon|  70000|    N1|
| Analyste|Marseille|  60000|    N2|
|  Manager| Toulouse|  90000|    N3|
|Ingénieur|Marseille|  75000|    N1|
|Stagiaire|    Paris|  32000|  NULL|
|Directeur|     NULL|   NULL|    N4|
+---------+---------+-------+------+



In [11]:
# Semi-jointure : lignes de gauche avec
# correspondance, sans colonnes de droite
df_gauche.join(df_droite, on="poste", how="semi").show()
# +---------+---------+-------+
# |    poste|    ville|salaire|
# +---------+---------+-------+
# |  Manager|    Paris|  80000|
# |Ingénieur|     Lyon|  70000|
# | Analyste|Marseille|  60000|
# |  Manager| Toulouse|  90000|
# |Ingénieur|Marseille|  75000|
# +---------+---------+-------+

# Anti-jointure : les lignes de gauche SANS correspondance
df_gauche.join(df_droite, on="poste", how="anti").show()
# +---------+-----+-------+
# |    poste|ville|salaire|
# +---------+-----+-------+
# |Stagiaire|Paris|  32000|
# +---------+-----+-------+

+---------+---------+-------+
|    poste|    ville|salaire|
+---------+---------+-------+
|  Manager|    Paris|  80000|
|Ingénieur|     Lyon|  70000|
| Analyste|Marseille|  60000|
|  Manager| Toulouse|  90000|
|Ingénieur|Marseille|  75000|
+---------+---------+-------+

+---------+-----+-------+
|    poste|ville|salaire|
+---------+-----+-------+
|Stagiaire|Paris|  32000|
+---------+-----+-------+



In [12]:
# Produit cartésien : chaque ligne de
# gauche avec chaque ligne de droite
produit = df_gauche.crossJoin(df_droite)
print("Nombre de lignes :", produit.count())
# Nombre de lignes : 24
produit.show(5)
# +-----+---------+-------+---------+------+
# |ville|    poste|salaire|    poste|niveau|
# +-----+---------+-------+---------+------+
# |Paris|  Manager|  80000|Directeur|    N4|
# |Paris|  Manager|  80000|  Manager|    N3|
# |Paris|  Manager|  80000| Analyste|    N2|
# |Paris|  Manager|  80000|Ingénieur|    N1|
# | Lyon|Ingénieur|  70000|Directeur|    N4|
# +-----+---------+-------+---------+------+
# only showing top 5 rows

Nombre de lignes : 24
+-----+---------+-------+---------+------+
|ville|    poste|salaire|    poste|niveau|
+-----+---------+-------+---------+------+
|Paris|  Manager|  80000|Directeur|    N4|
|Paris|  Manager|  80000|  Manager|    N3|
|Paris|  Manager|  80000| Analyste|    N2|
|Paris|  Manager|  80000|Ingénieur|    N1|
| Lyon|Ingénieur|  70000|Directeur|    N4|
+-----+---------+-------+---------+------+
only showing top 5 rows


## 4. Concaténer des DataFrames

`union()` empile par position, `unionByName()` par nom.

In [13]:
# Empiler deux DataFrames qui ont les colonnes dans le même ordre
data = [("A", 1, "P"), ("B", 2, "Q"), ("C", 3, "R"), ("D", 4, "S")]
df1 = spark.createDataFrame(data[:2], ["c1", "c2", "c3"])
df2 = spark.createDataFrame(data[2:], ["c1", "c2", "c3"])

df1.union(df2).show()
# +---+---+---+
# | c1| c2| c3|
# +---+---+---+
# |  A|  1|  P|
# |  B|  2|  Q|
# |  C|  3|  R|
# |  D|  4|  S|
# +---+---+---+

+---+---+---+
| c1| c2| c3|
+---+---+---+
|  A|  1|  P|
|  B|  2|  Q|
|  C|  3|  R|
|  D|  4|  S|
+---+---+---+



In [14]:
# Mêmes colonnes, mais dans un autre ordre
df3 = spark.createDataFrame(data[2:], ["c3", "c2", "c1"])

print("union() empile par position :")
# union() empile par position :
df1.union(df3).show()
# +---+---+---+
# | c1| c2| c3|
# +---+---+---+
# |  A|  1|  P|
# |  B|  2|  Q|
# |  C|  3|  R|
# |  D|  4|  S|
# +---+---+---+

print("unionByName() aligne les colonnes par nom :")
# unionByName() aligne les colonnes par nom :
df1.unionByName(df3).show()
# +---+---+---+
# | c1| c2| c3|
# +---+---+---+
# |  A|  1|  P|
# |  B|  2|  Q|
# |  R|  3|  C|
# |  S|  4|  D|
# +---+---+---+

# Une colonne manque d'un côté : elle est complétée par NULL
df4 = spark.createDataFrame([("E", 5)], ["c1", "c2"])
df1.unionByName(df4, allowMissingColumns=True).show()
# +---+---+----+
# | c1| c2|  c3|
# +---+---+----+
# |  A|  1|   P|
# |  B|  2|   Q|
# |  E|  5|NULL|
# +---+---+----+

union() empile par position :
+---+---+---+
| c1| c2| c3|
+---+---+---+
|  A|  1|  P|
|  B|  2|  Q|
|  C|  3|  R|
|  D|  4|  S|
+---+---+---+

unionByName() aligne les colonnes par nom :
+---+---+---+
| c1| c2| c3|
+---+---+---+
|  A|  1|  P|
|  B|  2|  Q|
|  R|  3|  C|
|  S|  4|  D|
+---+---+---+

+---+---+----+
| c1| c2|  c3|
+---+---+----+
|  A|  1|   P|
|  B|  2|   Q|
|  E|  5|NULL|
+---+---+----+



## 5. Les fonctions de fenêtre

Moyenne mobile, partitions, `rangeBetween()`, classements, `lag()`, `lead()`, `first()` et `last()`.

In [15]:
# Moyenne mobile sur 4 relevés : la ligne courante et les 3 précédentes
from pyspark.sql import Window

donnees_temp = [("2026-05-01", 25.0), ("2026-05-02", 26.5), ("2026-05-03", 24.8),
                ("2026-05-04", 28.3), ("2026-05-05", 27.2), ("2026-05-06", 26.7),
                ("2026-05-07", 25.4), ("2026-05-08", 24.9)]
df_temp = spark.createDataFrame(donnees_temp, "date STRING, temp DOUBLE")

fenetre = Window.orderBy("date").rowsBetween(-3, Window.currentRow)

df_temp.withColumn("moyenne_mobile",
                   F.round(F.avg("temp").over(fenetre), 2)).show()
# +----------+----+--------------+
# |      date|temp|moyenne_mobile|
# +----------+----+--------------+
# |2026-05-01|25.0|          25.0|
# |2026-05-02|26.5|         25.75|
# |2026-05-03|24.8|         25.43|
# |2026-05-04|28.3|         26.15|
# |2026-05-05|27.2|          26.7|
# |2026-05-06|26.7|         26.75|
# |2026-05-07|25.4|          26.9|
# |2026-05-08|24.9|         26.05|
# +----------+----+--------------+

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
 warnings.warn(


+----------+----+--------------+
|      date|temp|moyenne_mobile|
+----------+----+--------------+
|2026-05-01|25.0|          25.0|
|2026-05-02|26.5|         25.75|
|2026-05-03|24.8|         25.43|
|2026-05-04|28.3|         26.15|
|2026-05-05|27.2|          26.7|
|2026-05-06|26.7|         26.75|
|2026-05-07|25.4|          26.9|
|2026-05-08|24.9|         26.05|
+----------+----+--------------+



In [16]:
# Classement des salaires à l'intérieur de chaque poste
donnees_sal = [("Paris", "Manager", 80000), ("Lyon", "Ingénieur", 70000),
               ("Marseille", "Analyste", 60000), ("Toulouse", "Manager", 90000),
               ("Marseille", "Ingénieur", 75000), ("Paris", "Analyste", 65000)]
df_sal = spark.createDataFrame(donnees_sal,
                               "ville STRING, poste STRING, salaire INT")

# Ordre croissant (par défaut)
fenetre_poste = Window.partitionBy("poste").orderBy("salaire")
df_sal.withColumn("rang", F.row_number().over(fenetre_poste)).show()
# +---------+---------+-------+----+
# |    ville|    poste|salaire|rang|
# +---------+---------+-------+----+
# |Marseille| Analyste|  60000|   1|
# |    Paris| Analyste|  65000|   2|
# |     Lyon|Ingénieur|  70000|   1|
# |Marseille|Ingénieur|  75000|   2|
# |    Paris|  Manager|  80000|   1|
# | Toulouse|  Manager|  90000|   2|
# +---------+---------+-------+----+

# Ordre décroissant, et moyenne du poste à côté de chaque salaire
fenetre_desc = Window.partitionBy("poste").orderBy(F.col("salaire").desc())
(df_sal
   .withColumn("rang", F.row_number().over(fenetre_desc))
   .withColumn("moyenne_poste", F.avg("salaire").over(Window.partitionBy("poste")))
   .show())
# +---------+---------+-------+----+-------------+
# |    ville|    poste|salaire|rang|moyenne_poste|
# +---------+---------+-------+----+-------------+
# |    Paris| Analyste|  65000|   1|      62500.0|
# |Marseille| Analyste|  60000|   2|      62500.0|
# |Marseille|Ingénieur|  75000|   1|      72500.0|
# |     Lyon|Ingénieur|  70000|   2|      72500.0|
# | Toulouse|  Manager|  90000|   1|      85000.0|
# |    Paris|  Manager|  80000|   2|      85000.0|
# +---------+---------+-------+----+-------------+

+---------+---------+-------+----+
|    ville|    poste|salaire|rang|
+---------+---------+-------+----+
|Marseille| Analyste|  60000|   1|
|    Paris| Analyste|  65000|   2|
|     Lyon|Ingénieur|  70000|   1|
|Marseille|Ingénieur|  75000|   2|
|    Paris|  Manager|  80000|   1|
| Toulouse|  Manager|  90000|   2|
+---------+---------+-------+----+

+---------+---------+-------+----+-------------+
|    ville|    poste|salaire|rang|moyenne_poste|
+---------+---------+-------+----+-------------+
|    Paris| Analyste|  65000|   1|      62500.0|
|Marseille| Analyste|  60000|   2|      62500.0|
|Marseille|Ingénieur|  75000|   1|      72500.0|
|     Lyon|Ingénieur|  70000|   2|      72500.0|
| Toulouse|  Manager|  90000|   1|      85000.0|
|    Paris|  Manager|  80000|   2|      85000.0|
+---------+---------+-------+----+-------------+



In [17]:
# rangeBetween : la fenêtre porte sur les VALEURS de id, pas sur les positions
donnees_id = [(1, "a"), (1, "a"), (2, "a"), (1, "b"), (2, "b"), (3, "b")]
df_id = spark.createDataFrame(donnees_id, ["id", "categorie"])

fenetre_plage = (Window.partitionBy("categorie")
                       .orderBy("id")
                       .rangeBetween(Window.currentRow, 1))

df_id.withColumn("somme_glissante", F.sum("id").over(fenetre_plage)).show()
# +---+---------+---------------+
# | id|categorie|somme_glissante|
# +---+---------+---------------+
# |  1|        a|              4|
# |  1|        a|              4|
# |  2|        a|              2|
# |  1|        b|              3|
# |  2|        b|              5|
# |  3|        b|              3|
# +---+---------+---------------+

+---+---------+---------------+
| id|categorie|somme_glissante|
+---+---------+---------------+
|  1|        a|              4|
|  1|        a|              4|
|  2|        a|              2|
|  1|        b|              3|
|  2|        b|              5|
|  3|        b|              3|
+---+---------+---------------+



In [18]:
# rank, dense_rank et row_number côte à côte
fenetre_cat = Window.partitionBy("categorie").orderBy("id")

(df_id
   .withColumn("rank", F.rank().over(fenetre_cat))
   .withColumn("dense_rank", F.dense_rank().over(fenetre_cat))
   .withColumn("row_number", F.row_number().over(fenetre_cat))
   .show())
# +---+---------+----+----------+----------+
# | id|categorie|rank|dense_rank|row_number|
# +---+---------+----+----------+----------+
# |  1|        a|   1|         1|         1|
# |  1|        a|   1|         1|         2|
# |  2|        a|   3|         2|         3|
# |  1|        b|   1|         1|         1|
# |  2|        b|   2|         2|         2|
# |  3|        b|   3|         3|         3|
# +---+---------+----+----------+----------+

+---+---------+----+----------+----------+
| id|categorie|rank|dense_rank|row_number|
+---+---------+----+----------+----------+
|  1|        a|   1|         1|         1|
|  1|        a|   1|         1|         2|
|  2|        a|   3|         2|         3|
|  1|        b|   1|         1|         1|
|  2|        b|   2|         2|         2|
|  3|        b|   3|         3|         3|
+---+---------+----+----------+----------+



In [19]:
# lag : la valeur de la ligne précédente ; lead : celle de la ligne suivante
fenetre_date = Window.orderBy("date")

(df_temp
   .withColumn("lag_1", F.lag("temp", 1).over(fenetre_date))
   .withColumn("lead_1", F.lead("temp", 1).over(fenetre_date))
   .withColumn("variation_pct",
               F.round((F.col("temp") - F.col("lag_1")) / F.col("lag_1") * 100, 2))
   .show())
# +----------+----+-----+------+-------------+
# |      date|temp|lag_1|lead_1|variation_pct|
# +----------+----+-----+------+-------------+
# |2026-05-01|25.0| NULL|  26.5|         NULL|
# |2026-05-02|26.5| 25.0|  24.8|          6.0|
# |2026-05-03|24.8| 26.5|  28.3|        -6.42|
# |2026-05-04|28.3| 24.8|  27.2|        14.11|
# |2026-05-05|27.2| 28.3|  26.7|        -3.89|
# |2026-05-06|26.7| 27.2|  25.4|        -1.84|
# |2026-05-07|25.4| 26.7|  24.9|        -4.87|
# |2026-05-08|24.9| 25.4|  NULL|        -1.97|
# +----------+----+-----+------+-------------+

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
 warnings.warn(


+----------+----+-----+------+-------------+
|      date|temp|lag_1|lead_1|variation_pct|
+----------+----+-----+------+-------------+
|2026-05-01|25.0| NULL|  26.5|         NULL|
|2026-05-02|26.5| 25.0|  24.8|          6.0|
|2026-05-03|24.8| 26.5|  28.3|        -6.42|
|2026-05-04|28.3| 24.8|  27.2|        14.11|
|2026-05-05|27.2| 28.3|  26.7|        -3.89|
|2026-05-06|26.7| 27.2|  25.4|        -1.84|
|2026-05-07|25.4| 26.7|  24.9|        -4.87|
|2026-05-08|24.9| 25.4|  NULL|        -1.97|
+----------+----+-----+------+-------------+



In [20]:
# first et last dans chaque partition
donnees_fl = [(4, "a"), (7, "b"), (5, "a"), (-5, "b"), (2, "a"), (3, "b")]
df_fl = spark.createDataFrame(donnees_fl, ["id", "categorie"])

fenetre_cat = Window.partitionBy("categorie").orderBy("id")
fenetre_complete = fenetre_cat.rowsBetween(Window.unboundedPreceding,
                                           Window.unboundedFollowing)

(df_fl
   .withColumn("premier", F.first("id").over(fenetre_cat))
   .withColumn("dernier_piege", F.last("id").over(fenetre_cat))
   .withColumn("dernier", F.last("id").over(fenetre_complete))
   .show())
# +---+---------+-------+-------------+-------+
# | id|categorie|premier|dernier_piege|dernier|
# +---+---------+-------+-------------+-------+
# |  2|        a|      2|            2|      5|
# |  4|        a|      2|            4|      5|
# |  5|        a|      2|            5|      5|
# | -5|        b|     -5|           -5|      7|
# |  3|        b|     -5|            3|      7|
# |  7|        b|     -5|            7|      7|
# +---+---------+-------+-------------+-------+

+---+---------+-------+-------------+-------+
| id|categorie|premier|dernier_piege|dernier|
+---+---------+-------+-------------+-------+
|  2|        a|      2|            2|      5|
|  4|        a|      2|            4|      5|
|  5|        a|      2|            5|      5|
| -5|        b|     -5|           -5|      7|
|  3|        b|     -5|            3|      7|
|  7|        b|     -5|            7|      7|
+---+---------+-------+-------------+-------+



## 6. Gérer les valeurs manquantes

Compter, supprimer et remplir les valeurs manquantes avec `na`.

In [21]:
# Un DataFrame avec des valeurs manquantes
donnees_na = [("E1", "Pierre Morel", 20000, 5.3),
              ("E2", None, 40000, 9.2),
              ("E3", "Marc Joly", None, 3.5),
              ("E4", "Katia Chen", None, None)]
schema_na = "id_emp STRING, nom_complet STRING, salaire INT, credit DOUBLE"
df_na = spark.createDataFrame(donnees_na, schema_na)
df_na.show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# |    E2|        NULL|  40000|   9.2|
# |    E3|   Marc Joly|   NULL|   3.5|
# |    E4|  Katia Chen|   NULL|  NULL|
# +------+------------+-------+------+

# Nombre de valeurs manquantes par colonne
df_na.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
              for c in df_na.columns]).show()
# +------+-----------+-------+------+
# |id_emp|nom_complet|salaire|credit|
# +------+-----------+-------+------+
# |     0|          1|      2|     1|
# +------+-----------+-------+------+

print(type(df_na.na))
# <class 'pyspark.sql.connect.dataframe.DataFrameNaFunctions'>

+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
|    E2|        NULL|  40000|   9.2|
|    E3|   Marc Joly|   NULL|   3.5|
|    E4|  Katia Chen|   NULL|  NULL|
+------+------------+-------+------+

+------+-----------+-------+------+
|id_emp|nom_complet|salaire|credit|
+------+-----------+-------+------+
|     0|          1|      2|     1|
+------+-----------+-------+------+

<class 'pyspark.sql.connect.dataframe.DataFrameNaFunctions'>


In [22]:
# Supprimer les lignes incomplètes
# au moins un NULL : supprimée
df_na.na.drop().show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# +------+------------+-------+------+
# au moins 3 valeurs non nulles
df_na.na.drop(thresh=3).show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# |    E2|        NULL|  40000|   9.2|
# |    E3|   Marc Joly|   NULL|   3.5|
# +------+------------+-------+------+
# seulement les lignes 100 % NULL
df_na.na.drop(how="all").show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# |    E2|        NULL|  40000|   9.2|
# |    E3|   Marc Joly|   NULL|   3.5|
# |    E4|  Katia Chen|   NULL|  NULL|
# +------+------------+-------+------+
# ne regarde que nom_complet
df_na.na.drop(subset=["nom_complet"]).show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# |    E3|   Marc Joly|   NULL|   3.5|
# |    E4|  Katia Chen|   NULL|  NULL|
# +------+------------+-------+------+

+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
+------+------------+-------+------+

+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
|    E2|        NULL|  40000|   9.2|
|    E3|   Marc Joly|   NULL|   3.5|
+------+------------+-------+------+

+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
|    E2|        NULL|  40000|   9.2|
|    E3|   Marc Joly|   NULL|   3.5|
|    E4|  Katia Chen|   NULL|  NULL|
+------+------------+-------+------+

+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
|    E3|   Marc Joly|   NULL|   3.5|
|    E4|  Katia Chen|   NULL|  NULL|
+------+------------+-------+------

In [23]:
# Remplir les valeurs manquantes
# colonnes numériques seulement
df_na.na.fill(0).show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# |    E2|        NULL|  40000|   9.2|
# |    E3|   Marc Joly|      0|   3.5|
# |    E4|  Katia Chen|      0|   0.0|
# +------+------------+-------+------+
# colonnes texte seulement
df_na.na.fill("Inconnu").show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# |    E2|     Inconnu|  40000|   9.2|
# |    E3|   Marc Joly|   NULL|   3.5|
# |    E4|  Katia Chen|   NULL|  NULL|
# +------+------------+-------+------+
# explicite : recommandé
df_na.na.fill(0, subset=["salaire"]).show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# |    E2|        NULL|  40000|   9.2|
# |    E3|   Marc Joly|      0|   3.5|
# |    E4|  Katia Chen|      0|  NULL|
# +------+------------+-------+------+

+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
|    E2|        NULL|  40000|   9.2|
|    E3|   Marc Joly|      0|   3.5|
|    E4|  Katia Chen|      0|   0.0|
+------+------------+-------+------+

+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
|    E2|     Inconnu|  40000|   9.2|
|    E3|   Marc Joly|   NULL|   3.5|
|    E4|  Katia Chen|   NULL|  NULL|
+------+------------+-------+------+

+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
|    E2|        NULL|  40000|   9.2|
|    E3|   Marc Joly|      0|   3.5|
|    E4|  Katia Chen|      0|  NULL|
+------+------------+-------+------+



In [24]:
# Remplir par la moyenne de la colonne
salaire_moyen = df_na.agg(F.mean("salaire")).collect()[0][0]
print("Salaire moyen :", salaire_moyen)
# Salaire moyen : 30000.0

df_na.na.fill(salaire_moyen, subset=["salaire"]).show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# |    E2|        NULL|  40000|   9.2|
# |    E3|   Marc Joly|  30000|   3.5|
# |    E4|  Katia Chen|  30000|  NULL|
# +------+------------+-------+------+

# Une valeur différente par colonne, comme fillna() en Pandas
df_na.na.fill({"nom_complet": "Inconnu", "salaire": 0, "credit": 0.0}).show()
# +------+------------+-------+------+
# |id_emp| nom_complet|salaire|credit|
# +------+------------+-------+------+
# |    E1|Pierre Morel|  20000|   5.3|
# |    E2|     Inconnu|  40000|   9.2|
# |    E3|   Marc Joly|      0|   3.5|
# |    E4|  Katia Chen|      0|   0.0|
# +------+------------+-------+------+

Salaire moyen : 30000.0
+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
|    E2|        NULL|  40000|   9.2|
|    E3|   Marc Joly|  30000|   3.5|
|    E4|  Katia Chen|  30000|  NULL|
+------+------------+-------+------+

+------+------------+-------+------+
|id_emp| nom_complet|salaire|credit|
+------+------------+-------+------+
|    E1|Pierre Morel|  20000|   5.3|
|    E2|     Inconnu|  40000|   9.2|
|    E3|   Marc Joly|      0|   3.5|
|    E4|  Katia Chen|      0|   0.0|
+------+------------+-------+------+



## 7. Ce qu'il faut retenir

- `groupBy()` renvoie un GroupedData : `agg()` calcule les agrégations, `pivot()` construit un tableau croisé (donne-lui la liste des valeurs si tu la connais).
- `join()` couvre 7 types de jointures ; la semi-jointure et l'anti-jointure filtrent le DataFrame de gauche sans ajouter de colonnes.
- Préfère `unionByName()` à `union()`, qui empile par position sans jamais vérifier les noms.
- Une fonction de fenêtre garde toutes les lignes : `partitionBy()` découpe, `orderBy()` trie, `rowsBetween()` et `rangeBetween()` bornent. Une fenêtre triée s'arrête par défaut à la ligne courante.
- Compte tes valeurs manquantes, puis traite-les avec `na.drop()` ou `na.fill()`, en précisant `subset` pour rester explicite.

---

## Pour aller plus loin

- [Spark et Databricks : maîtriser colonnes, lignes et tris en PySpark (Partie 3)](https://www.monshotdata.com/p/spark-databricks-colonnes-lignes-tris-partie-3) — la partie précédente : sélectionner, transformer, filtrer et trier
- [Spark et Databricks : le Machine Learning à grande échelle avec MLlib (Partie 5)](https://www.monshotdata.com/p/spark-databricks-machine-learning-mllib-partie-5) — la partie suivante : MLlib, évaluateurs, pipelines et validation croisée